# EPU E6 real-data GPU study

Frozen before data access. This exploratory successor runs 20 repeats of five-fold stratified group cross-validation on the real Optical Digits dataset, with fixed same-bank controls, paired inference, Holm adjustment, and a shuffled-label control. It refuses CPU fallback and does not modify the retained E5-R1 ledger.


In [ ]:
from pathlib import Path
import base64, hashlib, io, json, tarfile
PAYLOAD_SHA256 = "05b38dc92c2fb2808de27874e7b35b86a11569da40228a545b9e9787f708f31a"
PAYLOAD_B64 = """H4sIAESj2GoA/+097XLbOJLz20+B49TdUBmKkeTPeEdT60mUmdRm4pztzGZKp+LSEmRzTJEakrKtybnqHuKe8J7k+gMgAYqynfnI7tYKVYlIAmg0Go1Go7sB+0/9p39+G95+J8OJzD77Q1KH07rfTmd7u3rG791Or9v9TNz+MejYaZEXYQbNf4q2/gFTb1/Mimgm+939Z73dg97OTs/f7nb3u3tbf2/UNukTJDlfBHLv6duT47Pj58ev/Z/yNPm928BJvbezQ7/7e7s813s7es7v7u/sftbd7e3tdXvdnd0ezP/t7W7nM/FJ5qSe/9kyvLfjUGw6/RQIfdr0YUsI51pmeZQmzqFwBm/ftU8GR69fHJ0dtQd77W/xvet4WGyapb/IJFgU42ASFhKL9zq9vXa30+6pIvI6mshkLIMsjanA4P3b18cnR2fHJz8GCDdAwMHpu+fPB6enxyfBqzcvj0++H7wIvvkxgLaD/zzqBkdvXgQ/dIMfdhkmtBXmsgBoiCx8wCaKqFgi/HwcXUVFO5Zhlog4DSfBJLqIilyk8yIah7G4DJPJTRYVhUwEZxFUxPV2LseFnACuNznAAgm4X8+ayrBYZBKz93bqmeM4zHPKG3Y80fVEzxPbntjxxK4n9jyx74kDTzwbqXpxNJZJTlT55vRFe7v9PA4XuRTYP0AtL7LofAFwxU1UXAqzY38SsDZH15AFQzAT756/EseqeydynF4kUQHDJ9Kp+M7o7Qurt/hVZkE0yYPwOozi8JwGaBrGuYQSdzx80LUMVoOkCONgAWAR2UzOgQoIfQo4tKdpPBGAbFhE0whQusjSxVyMszTP29dhHEF3kJcIHldFCvU6zEJQGV936S2Xkt6G3Q5Srwv/bXeAajv4utsBWu7h0z7+d9B5BrTE3G4HH7tdzO/2MK+7Tf/v0PddBNOlit19+nJAz1S3B8oGjYeTpFEuA5jVySTMgGuAcwlxHk4fBrTjd3fxf1VhJovLlNF1JlEGLBDMFnEROZ5w0gxonQDNinR+hR/CZHyZZuXrVZJs42+cXmTywlEA04mMK66eZ2mRFsu5zAMYBGYuJJxXyw6mPCwTCeM5ixJgnGgM45gVlzIv2tMoy2GwEgEjBJnJhUD2hg/xUrNCXsg5gj5Q72E8vwzhvePvqS/nssAPe35HfSg7mElgUwnjHMTh7HyCpbplKdXrhjJAS1UGSBFcwZdtPSuIJMFzC5D6OAtvA+Ra4p9Op2TTKJnKDMVMRb1xmkxZ9FBrz3Z1T9K0QGadA1Z5OJvHNF+7uO5owoYwmJMgjy4SJPxsUZR8QMV0uWk4i+LlDXJNRbCObodYYQ5TXMml79J4JsLJT7C84HQSIU0QUVxmUsLLeJGF46W4XM5THDZZTlN70jlll42SwIBUlntO8pBaRHH97uzV61dnPypoTBfofU4wDaYUwDiLXJh8LEJYBsILmNEK14kEHgTSCJoqIpbXMs4ryMh6s8UskNMpwGBqdMtcqBzpNYVLiK/7XATE9I3MxLPdfxdMe1EOEjwtkon4WsD0Qwq2mYJQJE1kO4cBnghjjMRcfEWD4FCzd95aupwcf/Pu9OwNrDm/jjSFIgJIg84/KQneHL+BxXZw8ur4pP36+NuTwbePJcU0uoVGYU6yqIGZCXMJe/bRLBNmIEWgkfYjCEVlBIjnxxCLCj9IgtOzo2/umR+WFFd9t6ghwwRWPlB9cLmex+FY4ty+lx8eZodH9xGKqh7C/2oRkxfABdcyoG6kxmoykSjQWEUT9lKhF4Y26AAyNpkpF1Ei1HLfEbTQd5D3f5FZykPqlEIzzwMUZvkllAq0QKMu9yqhBSvrDfBDhVYmf15g9xCx5+9eHAnQL1nfgWZA14Gl60B8G30jYK0D7WYmZ2lWrltTUFtAGQtoCEiNen38/C+gO745Do5efP/q7AyeAWCFZARKWjqbg1YVhAtYvrPoF2rb1nqAZ6NZQGQOM5Leg9t5nIKGA40DPcK4zToasnM6J3muFV1fnF2yCgcqKiibOdBJsK4lWE8FJSnLPZGnIPwjoDCWKFSRNvDRT2kE8Gj9ymY0DB4MwwRGIcH6OB5ZOlmMOef0+OyobNwTaSYWCYwCLLiokgpgLFALYWUp8XO27u7byfsb+8/G/lO3/+zs7e32Nvaff4Wk7D+wNX/x/cCfTf6INh6w/8COrW7/6e30ehv7z6dIn4sBrMDVGjfYoyU5X4zHoOOl2dbWGa5auZyHsB7KeCmUsQgWm7KQyBagO4SCDUSwKi0KWHZl217GVBNbsMhl0S1uUf8SXlzEEqrBnghbVWu1L14VYpJKXilhmxxNl7jS5QgLVlEJwArQYeRka7DbPgEFqijkbI4KxORCZrwk0355nMbiJkT0syiMcfEX53KaZpLX4qoDKYAGcEW1lvvYcwnq3mKyFAvYd8EWVOlG+UfaQjze9m2xHj1Os2wxJ92d9WPQDUIg1nmYXKldB2mE8LDIMsm6ASp3UHC6iGMquHX15o2HFLdU87ahmiMtQ1JhPK1SMvZtalWUe2hvi/aq5gZWt+gRIqHILxfTKRBXaYxa69TFfEGqHBA03EKVD/a9s4ixUPoeqky+OBLPYZBB94rPw/EVlp/IODqXirEyOQU6TxThtYGLrVjQhfEV7DQEKN1XeYOC5YvBtUwQVx5ToBRyJTSyJR+tzHkPqmbAhlvrlDP/n3HF/MfQ/3ZW9b/eRv/7FKl3UNP/9nZ8WKWf7W/v/DOy8yZ9ZFL6Hy1z/nz5h7Rxv/63293b3yv1v12a/729/Z2N/vcpkuM4L0lrIz0QtL9qhSQ9kPhCnIKCZrmIfai3tUXeqCCYLtBHFgQims3TDLUXWEbZoLS1pb9lF6CO5FK/j/Nr/XjxSzTXz5dhfgkqgX7FpvQzLMCX+jnNue05fIPiuuG3RpF5HBag6c30ewaqTFq+gUZTRHH5tsz1I8rCEulkMZsvBdp0SgTzK/LIcfPqxVcqY67xMPyQdsEYlNYwC8jzowu/VsrbSam72XVm6E0Zl7C1qS3IQZEEjeU8jENQXSobnM6YdvmpBg2bDnIZS9JZNNTTUof9FlXYv7wExdauOI/mEtEX5XBcyUB/rBUlrQi1MHQ/lS2wp+0UlDqZleROs/Hl1tbWD4OT01fHb0R/rQd66+T4+AzycYxd4LkoBo5r+UCyNL6WbssH9gKNDGBN5BTGN+zt7gXny0LmLmjiC3ko6KUl2l+jxn5IJkLYRyyyRHOdz7W4fMu/lLcwhjIv3JYNFdt2kfUOCZtGkBYCWBYwBabg95YGSKpmkMgbA5wnFL7p+U8wSAT9Dey3GDyB4q76syvYLrj8kvfPsgWMurwFZgrSK3ptURUyr1I93OW4zq0DxZIxbKuSi76zKKbtA6eFXI6u8li1g4km+gTmAFPEUwXYPJkU/R5aNbMiuJJL3Tzo9ulNkIRJ/yWaWFslLK7qU4dd578SR5OA5orerLnU20k0LhQxwxsYcpeG/qlwLBHktCySMnX0pq/PyCPw3AUonB1NyxL+hSzcMvChJf6tLzQPwnbQLqW92FQKdoErBdirTdm7FfmyEH0gPyDpBlmWZq6zSHBnVKChVgNwSsxiGBsbLLvHPTEctQi2zh2WGI3uaw2rw+ZlAdsU2EHDbi/n8AFl4Kcc1bxiWt2AGhq1eQsu5ov6wAC+uFWi6euPF5PQj4y4ArdVx+sE2gLJqjBbY7c/5H0kUKkMUCiHdZ7DmBrNAYHIZz+WyDxzmRURcEGHy7PzoA98WrhU16cvAbsTSopzsa/EgXgiYMux8+TJ9q/E2/RWCNgipzfsxnA0PiXeeYl3hSxmzsJkEaJglhMXg2pwC7RS2SgUwEyrFVSD+KHsgsMNoUMDqx9Wnkt0HYUzzGHy4IuRZ1KL55ej+mgUQpD0GdFTE4nQtABBZlkoCFSxIPCUI4sZDZ3ewRhEisyCREmb/BBWXR+XjCxcEvdVrzxKXAyGGTLCnDJUVU9MMFKiDxlTEAHF3g5TiJuAWdHXldvqwUffnhuC8Ox3PXEl5XwSzXJDiCagSqi2YvQRXvj4xdUQQfKtr4uTBVBMli5D+aovurLd7a3MEnP2oqLQHoMKBetmgTEcYkx6FbAXIcPmLHv+lv17yvia5KUhKEnLA3ImExDgRFzzw+GjiAWdNPrbQCoGqah1DUtZCqNrE20tDCDZeQrrAcNoIN1H0K6gTv02Ek4WaBoLYbEmE1/uvn+QPWfhfA4r7CFJzSHNIlw5ixGQ5sOdajNfxAW8D0f0PlVoRonJ0+8b2LnqPSy+AABqQXks5TpfTQ9g0Rin86VahEH6mYukoi9WRGJAYxrVMtvAfwjlEGVcntSnCgx3wIfPMpm4Zg2LqEZnuIbRI6AITk8m8xQYtYqDsonsiaX9WkZJHSJZAcUeczIMlRyaBatntV6+T6B0Tei8NzQIat0TZO3MaXg8c4jYCgp0Q+0H5DXAB2A/L6S7RFrHEaqMxhDN0YyIcN4nQxisPgMYlfkUV9BXxQxJ1KkIzVFdvKIhNbOLWXjr1oSRaqdNALVEarUqKOPLNJfY1JDgVRjcXIJKS0PMRVqwKlbktdgCAxVxw6GE4SxKXCufJMDch0LjK3e4DkN+GKIieTsqUSXq0jekLqNS5Xor7SiZa31vNWM7VNAA63bXKsIZmodtApfVTSpWLOLL2wIrWb1p6kRVmXlKVxxic/SpNQIlpCQ5l3+7yqXGTDI5dUU+WNPvrWfOQMZg3QyswgxhF4eSi0cXmDuzp19ttqzmcTur33EHeoj7JX5lHUWvSPzmbT04k1mUMmv+XC42YR6wtHdrdCP8W7rTXFjRylMo9PlHiYEGkKsEfxgOEKEBkj0Eap1Mk4tGEBFG95I6+7M/hlLSLZdIJCXKEzsetZqvHKsEFcsvqCmrUB9YSekzYGLs+eqxm1jM71iN2cGuVB9jk+5pFWdEQLFFYXIh3YOWLVJuZHRxWVQqQ55OC5x9e9DwE+EyvD+LuX/WUiqDPckVzqSqaFhQ3CqjkTJ0oY6/B+A540vowQ68rXT/SQW91UzFAzUTaLagSYXk+1CNOcsRowu6Byx8zTiy+1QmBtDWzGCRody8aQz8MW7YfLJeuUSyqpF6XtUVJQAwVLmc+6vTnqPH7E8NM71xWjeqSX/A5L19GCT14iNALn/rLIa5UoLAeeP+DOxwi+xw1d8uFeEY+EXmhTKmsHrBWjFIaRgRmfPOJy1kNV2SxJ8uEhr5MPZBQASXaeEugQPnsHjCSOuDCv1up+Xni5m7yjwWtxJ4zWI2wygmwVjAZV09owDBQ0FkBORhk0qKWeOoR1MVIgoCpWNJn5q+2KT8+qjbKimYJResiLB9FzhtGoIUDOC7iyjU9dBxHFmraBN8kAZQ3ectgktnAQhZT7xHE+FctvBYAB758DuaIlWc+Tgqd1moCQ4JKgxEGX9OVKkTiNc6RTuuwmQhND9mk/s4ikyy8AZZCDsKOgraI9Qw5rAp6rslth6phbzNaWEvSJQopjR2zRahCSEk888L2IGhuZTaI7L1dhHOPSWe7e+WBlIjODWYN9PVjF+tkxbZP44VTwoKviZq0/vH0xcEMMKjeul5LjNec7gvVJcp0vqYkWALHeBGb+dhMb6Et10+78BrJ7qoimr97Nid9riSsajiEAIM1M0pyy4PvSCALaP8RVJjBoMcw3ZXMfuopdkD/7c4w9hkUH++LPcqKHBcbuEJU9vmG4yC1sS09VZXgRJQ6KlwrT7gR8Ujl2k8CzhK3p1rFqFNNyqZajKV9kvzI1MszSbKzqH2cwqIHxVyluMiCRvavtIH8Nsh/T/slptc3tJmi4TO2+D2njid1V21d1attKqNfphcecIF2Mri38IRliBoKSCmrGBIx7IBlNLqzaNhphFyZzC0CLYFlGaQ9a263s2ryia1uYCiKTud8GScy+ZiNZ3Q0q1nliXu+XQU6/WgIyzQm1HX/I1UKSs1tcFQVcwc8d/kA8GpPU6TSa7arbsLViyg6hjNoTJ6G6ZJ7AlGgsOPaRWlIPdDJe+r79w9yOAHs4ZD9HBxjKnfLWOj6hiB8UpKWF46ruEZ5LBqr/j2SjBrvH4PwJvhMY1g2i3BaB/haj1Pn+3ocy3HBgTzN7DOQBzS+OC6bqm0Uc7faZNQynxlP6zKWcBlHM5zPJHFQw2g1ZNtND6XyfhyFmZX5J9Yq29W3BGWGhPK48R1e52dAzwSuHPQatTZzj+2gmnnXwJ2WZqAnFSaCslcEjPoUfBBmk0DEnAyUyXGl3J8BeISiiDRSkmxZutkFA9BmTx/GAVF2WYU9LIgJywVrT0OCJQe/oOuP3mybU22hvFSXzwYyxxaCvGwbAGjT6cODcBPS4yeiq7s9mwx4egOluyqP+Bx286I5DOstZohchlPQTdH45s98IZXsOZtXOtHGA47Pi55MNrwb9tHA+C2esOvHfrSUV/4F79Ub1RmVFoB2JBotoB9oFPDeh0x7QvzskrNJKp1E8633ZpskyINFd2FLmxlQGdBb6WyOmnbJOaa7ddt6UewW8hwiJQ9vYTOEZ/huBB4Fqe0nbNNHJBdMZNDh6+VKVBjzr/DDhCnRJ9LwzfETL20uyPEHeVqLguXv1bmVerEzkOYlwgxVFw9LczLw3V9S4344IT6NB9IYX6EXZoz5gNOd5VfJy2Eq4EModYIXRTVh/P6h7EzWnFd1JGunx21MC6nHZoEFjihnLdHp6fB6eD1y+BscHqG54w1hwcchoAi1AhdWONOr82+6lKAyu+sv430nAM9wk0XxXxRGLEQj555XPP+OAYjjqAmQfE/BUgfNEPNxnRaGxbGUpYr76t2idoAYMT0suKgntS0yKioijJ0g3uxSlCv7P76GuWo6SqGEAP9Fqu9D2Bj5JEFwogrcpkPgvfB0nabUXFDCNCtBmKPBcG9DgssTo+81HQ79zvZ9Jk3fRuDIIcb2lWK3GbZ97bkUx1a3V3BIrCntObajmy5aq6mkQUUAo7RQe3dDLlhMpguMcMNBruHZZUXWXkPSTQCXDWPFACF3zHvkADl0ro2gk4YqpsiagReUYydcZjAkj0O44C6Xs1go7MAsY6XoYVWo8tZraZmVKwWSgYVtWV45+/W8isGZZ0OSm5VFFjP34M3P7w6OX7z/eBNWcfQzudLUKZJwqhgPZ+/aExw7+XoPKuYenBNvZFMVA7tGsxIgwf66j0YpUDAjaOs+tHjGIjgOsqj81iqsA5ScXJfJtcR6F4cyINhLcEPr05fffN6ELwY/PDq+eBUy9s7Nc1VmJ8ybqAIHVWOYd69BHh0RIJs5Y0sm4x4W6ZsbkUGSggBQO+I5WkmAGpj1lj9/pr2xuzh9hmkAQfkZ1BtLwKyS5DWxRMeJlWAMWrwoeKek6O/EtfE/sUvfPSZYtkwYJRj2XQtGIrboiGsDXap8gYDFPsUcoZBblClkmq0D+edIRmN7B14te5xJNao5rcYXwO2TZGTbhIAtUBK9ysQHCMG5FGHadQ6x6YhPtLcr4xCJoK0RRVssg7IsQg7bJAC/FyzGoyvfWoaDZsgNbUEOKwLAFwqKrVqaAAftcR/WFllU/X+6/SQ6qWkoTodBTPwqroxIKTuOa0VwO8Dwgl6wQ/oPrfQhBzrvQkCycol/XL9qi9UvXpdqW1p46VTdJ1aXsd2tT98pQnZbozA2VXfOSY7VtZt8HxjWg0cdp/3yRCk703p460p6EeIQaT1nfh8eoHxhKs8t9rAug74U+StB3uLfMsG/igx9Mf1l+6MmllLVZCk9eECVwvVnTofVNjm3dMPPJPhAVkKfqjyneOTVABV0QrpHR5290Z4UdAq8pjI54RKCHs6FCspN0CFVXPlSs5ZlkgjDqWSi/VkCto1tZUx7B4goBqDFGwwg66vgiPGxi0cMvfjrjhaIxgYlfHVPTaP+8lnGwQ9EYj+ugiFdYkG0rt3OmvrpbbprIW3HuV7zS739zEfcusj6pv+uraixR9GVfP7PYNBfGFUazYE4aCtBbJmJf9SBxSsNvpRHGBShi/OQjxX3dLeyhKhx68Z7kcNkSZUhcBHEsoepgpMadj7e1JVX0NGW0qS6hZlHyCJUfu3EcUAtJYspuKKwdPKh+a6Wm0jWdyiIHkdVcULfrWxfYS8q5Yofc3bmgXJpqU1Zctptbai0vHLeMm6k8VTup5aZrRUWqrVx5CL94q85tQoOLy6SFhDLUz2JmT9YGjE7UG5F2PXJGlfdbilt+i86T+oTBW/QkDXHBOVCFT38VQnbB7ovhVM83vSAFMjig93e33XKVw1HWPE0EWcnsPyXUYmVnuGSg1uil5dpQNwAKpjH+7FZ8X1ZrvcBFpM4MXE6n6urvnmhEOLt7J7MMcMqaujNfpyCciYsngzlXZd8mmRldmsgT4sMe4eaLfmMCsbZudG49jrxj9uvv82Xe6ebjSrjpjQ2saHu8pTY2ijRFZ5+JSY+FKoI2EEiidaeUmiyWkA5LDmR1SObup05UyvbjIrBZZy529VHVEFVryZv7lJS0istkscnIv+43ZG7LqjG+Msjb5uZ6mMVdZVkPq6yJI5rK4Ma9QeurjVqLPECAML1hS0mKvF4c8JHWGgXtZXFDK5VA7NhjNtVfmRV+uTcY1jvVtN6HX87cf2hIvW+vL7Id5w+eIfPS5Ku/qDR6S6UvFx/VmZayW+tlDCrt0n9B4EV6PT70+Du8puuSov7Lg1NXkp+MYzJn11j+1ouP4e2hHegNzpgJCMjAMheMiAIpPstdyt3QPbcP/pepZsmUJqjWuUOly67/Cap9w+wKQslPrC2ihfZ1Q1rrQdrRxoqqoP8QbPaiqqqzP7Nf4ySWyMLV2r6YkFKNwZuvPMKM1HjUrT/cEjvIiaB4QssGVr6lRVhO7Zh4bhztaw8EJNonV1NahJgtrNoiPjztCv8Y5OvKZJ+5gpxIsvha2oFivcMIB/rUT6KIz4Llcbk6YCBg65/P06fV/BshHNoXrHZausfDUsLeBOeWUrP3jmzdHPdlG4maw0qvnKaVKYUx7VXvxG/h9oAR1ZNIswR49UlWlDsz37DsV3MKUojMp5efTqNbKUkoGKj0vcFUeXMLWH6XPxkm4Le+A2L5GWd8Diva+8J1WPbNLFZ59AKsPsiqNk2OExeISHZPeRHpE17g800cLuykVz9zrnh46zpe6ZeqUhtcwrlctloNcxg2oV2aCe7Xgww9ExNQf8YkAvigyjoZavYLoatnHIDGOAGvwMNRdIQ0U6WBuss5XWfCDUTF3xWbWj1Wmn51NjTOOy1gQGsRoi+SFDV/1OY1u1VW1F5Geto1WPUFwJm9SrRb2iFX9YXmtshb6sXLVMS3bzLciWhlCvONTzG5X2aoavFrO7QHFEq4UqZEemdNiqvKWGN/7l8esXwfeDs5NXz0/9cX7ttIwrQUpP6WOvB4EZHU9o8dd7fb3H9/Rm3SsjZ/ET7TR1f7ym8NamnWcVteo1BZ56qwGjxrlWviewj9cN+S+A4/5KH1x9kQl1AS8/yPvcm1atKu9pL+lyPLf1J+srhnwoG56qly9msOrZO1Yjjkp5qsxQ19qfS6nYzc4x1e7qz6GoJyNPr3cYg6AezQCJ1dvC65+M0s3zFE02jRn3BvAa8VpVmKl5l0T9Bu6KELUsa2o1RZx89+Pb47PvBqeD0zKkikdFhWICixPPVo1/3nz1FcfDo7/Ucbz7EML8alI4L1RUEDsaD8XfPhiRO3d/863C2JBh+1S0PBQfmol85yt0nM8/F28z2c7ncsxXj5ZDX+Fj6OSFnOGypgtVk5joUYpzpy2ePPlApxu+iCZfjO7E//3P/wr1gcUWfHzyRKsaOovfvhgd+nvTuz8Jp3nnNnXwfnt1C2l1v32EjoPrMBZDBc3UvL5AXYLAeqI5u8vZI98xR/hLFEuKUG9qGo5jD5lG7tTWi+hUfqksfFgRvV9Y8ln1/bCpYEk4Hj0eFoNlTwan716fnfqzCQhk5ukCdRq0ifl4B6lLfWo1yGUCNQuTaIpLLkZ35pWLl/QajKmpripQEZcZml1d54ljGnpJzSwu8SYfChNtkb5Nn1BEYnig8/3Rm1cvy3DF+oURJhpoo9IXX8VMkCJV7bdaoyper7xNy+7MQ0LUVJOfH3//9vXgbEALUMR/4sXGxpA1ebrI+C/HOPrWw1p8rH3FGKrudmDno6JpDSuqw3ZQ/jNIeNfQ/rN9+EGD068Xsr+jyLQH1StJZwUcKymqzweBEuquXE2W5bTY6rv+/KPsYoGL9FvK0Xd00YsfTmDdV/mu025j4GsbFUdAIKRO92F8QJ9E3195EVRzXe4KVKQY0bclL0EZsrJyLfrBenl1gh7f/DLm1riyI0NvgGHKNuJy6/Zs65AV0soCrsisDsIYLTBSkoOldB+wnP4bGWKRAOfmwqANZuMdyfpeqWxpnp1Vl8lgTLbRdGt9p/TJtcff56bOkdyO5bwQA/qhvwOT4zdLlJidJ/NE9e5TcHfu1pxM6s97UDjt/fMdVdzBC1L9kHwBDjtqGegsBDxafhCgxKLgTljt8vACs1EgYe5dU6s6+NBE+ym39O5kwDPDqqeC/83qa/plTzuzgqerGxyEsXQwyQC87gTZbIIAp1wQOPpyIZx/mxt6N2mTNmmTNmmTNmmTNmmTNmmTNmmTNmmTNmmTNmmTNmmTNmmTNmmTNmmTNmmTNmmTNmmTNmmTNmmTNmmT/uXS/wOQPFyMAKAAAA=="""
payload = base64.b64decode(PAYLOAD_B64)
assert hashlib.sha256(payload).hexdigest() == PAYLOAD_SHA256
root = Path("/kaggle/working/epu-e6-source")
root.mkdir(parents=True, exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(payload), mode="r:gz") as archive:
    archive.extractall(root, filter="data")
print(json.dumps({"status": "PASS_EXACT_SOURCE", "payload_sha256": PAYLOAD_SHA256, "bytes": len(payload)}, indent=2))


In [ ]:
import json, os, subprocess, sys
source = root / "epu_e6"
output = Path("/kaggle/working/epu-e6-realdata-gpu-r1")
print("Free-compute policy: paid compute is not authorized.")
subprocess.run([sys.executable, str(source / "study.py"), "--self-test"], cwd=source, check=True)
subprocess.run([sys.executable, str(source / "study.py"), "--output", str(output)], cwd=source, check=True)


In [ ]:
import json, shutil
output = Path("/kaggle/working/epu-e6-realdata-gpu-r1")
manifest = json.loads((output / "MANIFEST.json").read_text())
hypotheses = json.loads((output / "HYPOTHESES.json").read_text())
archive = shutil.make_archive("/kaggle/working/epu-e6-realdata-gpu-r1-evidence", "gztar", output)
print(json.dumps({
    "status": manifest["status"],
    "version": manifest["version"],
    "records": manifest["records"],
    "gpu_prediction_batches": manifest["gpu_prediction_batches"],
    "outcomes": hypotheses["outcomes"],
    "negative_control": hypotheses["negative_control"],
    "claim_boundary": manifest["claim_boundary"],
    "downloadable_archive": archive,
}, indent=2, sort_keys=True))
